# 04. Слияние видимости: НСУ-3 + БВС-4 + БВС-5 (угрозы №1, 5, 14, 15 как «деградация видимости»)

Видимость - общий знаменатель аэрозольных угроз (пыль, дым, мгла, облачность/туман): от неё зависит, **насколько можно доверять камере и lidar**, и именно её нужно оценить, **когда нет видеопотока**. Единого прямого измерения в ТЗ нет; есть четыре неполных следа:

| Источник | Что говорит | Слабость |
|---|---|---|
| НСУ-3 `visibility_m` | внешняя оценка | сглаженный прогноз: **занижает редкие локальные события** (в тумане завышает видимость в разы); данные старше 30 с устарели |
| БВС-4: дальность первого обнаружения реального препятствия lidar/camera относительно radar | прямой физический след потери дальности | событийный: только когда встретилось препятствие; нужна калибровка сенсора (дальность ↔ видимость) |
| БВС-4: «фантомы» lidar, статусы, молчание | пыль, дым, осадки | косвенно, зависит от типа помехи |
| БВС-5: T, RH (спред T − T_d) | туман/облако, сухой воздух при пыли | в реальности связь с видимостью слабее, чем в симуляторе |

## Что делаем
1. Оцениваем `ln V` на каждом тике БВС-5 (1 Гц) моделями с разными наборами источников и сравниваем с физическим слиянием (НСУ-3 + оценка по дальностям).
2. Оцениваем **событие «V < 1 км»** и **«V < 0,3 км»** (задержка, ложные тревоги).
3. Проверяем **смену условий**, а именно то, чего честно нельзя знать заранее: (а) связь влажности с видимостью ослабла до 0,3, (б) сенсоры слепнут раньше, чем мы предполагаем (дальности ×0,7), (в) прогноз НСУ-3 хуже (σ 1,0 вместо 0,6).
4. Выход: `V̂` с оценкой неопределённости, класс видимости, **доверие сенсорам**.

> **Честность оценки.** Связь дальностей сенсоров с видимостью (`lidar = 0,15·V`, `camera = 0,4·V`), форма ошибки прогноза НСУ-3 и связь влажности с видимостью - допущения генератора (ноутбук 00). Основная метрика - ошибка по диапазонам видимости, потому что 90 % времени - ясная погода, и общая медиана ошибок ≈ 0 ничего не говорит.

In [ ]:
import sys, pathlib, warnings, time
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src").exists())
sys.path.insert(0, str(ROOT)); warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

from src import streams as st
from src.eval_bvs4 import KINDS4, build_cache4
from src.eval_vis import (build_rows_vis, ln_metrics, by_bin, physics_fusion, events_vis, VLAB)
from src.detectors_vis import VisFeatures, VIS_FEATS, physical_range_estimate
plt.rcParams["figure.dpi"] = 110; pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
t0 = time.time()
train = build_cache4(KINDS4, range(300, 312)); Xtr, ytr, _ = build_rows_vis(train); del train
test = build_cache4(KINDS4, range(400, 408)); Xte, yte, mte = build_rows_vis(test)
print("строк: обучение", Xtr.shape, "| тест", Xte.shape, "| время", round(time.time() - t0), "с")
JN = VIS_FEATS.index("logV")
sets = {"БВС-5": list(range(14)), "БВС-5 + НСУ-3": list(range(18)), "БВС-5 + НСУ-3 + БВС-4 (окно, контекст)": list(range(47)), "+ дальности первого обнаружения": list(range(50))}
models = {n: HistGradientBoostingRegressor(max_iter=200, random_state=0).fit(Xtr[:, cols], ytr) for n, cols in sets.items()}
print("доли времени по диапазонам видимости (тест):", pd.Series(pd.cut(np.exp(yte), [0, 300, 1000, 3000, 10000, 1e9], labels=VLAB)).value_counts(normalize=True).round(3).to_dict())

## 1. Что говорит каждый источник
Слева: прогноз НСУ-3 против истины по диапазонам; в середине: физическая оценка видимости по дальности первого обнаружения (`V = d/k`, только когда ни один сенсор не в насыщении); справа: **доступность** этой оценки - доля тиков, когда есть свежий трек препятствия.

In [ ]:
vb = [0, 300, 1000, 3000, 10000, 1e9]; Vt = np.exp(yte); bi = np.digitize(Vt, vb[1:-1])
j_l, j_c, j_a = (VIS_FEATS.index(k) for k in ("d_first_lidar", "d_first_cam", "age_ratio"))
est = [physical_range_estimate(x[j_l], x[j_c], x[j_a]) for x in Xte]; vphys = np.array([e[0] for e in est]); okp = np.array([e[1] for e in est])
fig, axs = plt.subplots(1, 3, figsize=(14, 3.8))
for a, (ttl, arr, mask) in zip(axs[:2], (("НСУ-3: оценка / истина", np.exp(Xte[:, JN]), np.ones(len(Xte), bool)), ("по дальностям сенсоров (надёжные)", vphys, okp))):
    med, q1, q3, xs = [], [], [], []
    for i, lab in enumerate(VLAB):
        m = (bi == i) & mask & np.isfinite(arr)
        if m.sum() > 20: r = arr[m] / Vt[m]; med.append(np.median(r)); q1.append(np.percentile(r, 25)); q3.append(np.percentile(r, 75)); xs.append(i)
    a.errorbar(xs, med, yerr=[np.array(med) - np.array(q1), np.array(q3) - np.array(med)], fmt="o-"); a.axhline(1, c="k", ls=":"); a.set_yscale("log"); a.set_title(ttl, fontsize=9)
    a.set_xticks(range(5), VLAB, fontsize=8); a.set_ylabel("оценка / истина (медиана, IQR)"); a.grid(alpha=.3)
avail = [okp[bi == i].mean() for i in range(5)]; axs[2].bar(range(5), avail); axs[2].set_xticks(range(5), VLAB, fontsize=8); axs[2].set_ylabel("доля тиков с надёжной оценкой"); axs[2].set_title("доступность оценки по дальностям", fontsize=9); axs[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 2. Сравнение оценок видимости
Метрика: медианная абсолютная ошибка в лог-шкале `|ln(V̂/V)|` по диапазонам видимости (0,22 ≈ ×1,25; 0,69 = ×2; 2,3 = ×10), по тестовым сценариям; обучение и тест - разные зёрна. «Физическое слияние» = НСУ-3 (σ 0,7) + оценка по дальностям (σ 0,3, если надёжна), байесовски в лог-шкале.

In [ ]:
res = {"НСУ-3 (как есть)": by_bin(Xte[:, JN], yte), "физическое слияние (НСУ-3 + дальности)": by_bin(physics_fusion(Xte), yte)}
for n, cols in sets.items(): res["GBM: " + n] = by_bin(models[n].predict(Xte[:, cols]), yte)
tab = pd.DataFrame(res).T[VLAB]; tab["среднее по диапазонам"] = tab.mean(axis=1); display(tab.round(2))
m_low = np.exp(yte) < 1000
rows = [dict(метод=k, **ln_metrics(v[m_low], yte[m_low])) for k, v in {"НСУ-3 (как есть)": Xte[:, JN], "GBM: БВС-5": models["БВС-5"].predict(Xte[:, sets["БВС-5"]]),
        "GBM: БВС-5 + НСУ-3": models["БВС-5 + НСУ-3"].predict(Xte[:, sets["БВС-5 + НСУ-3"]]), "GBM: + БВС-4": models["БВС-5 + НСУ-3 + БВС-4 (окно, контекст)"].predict(Xte[:, sets["БВС-5 + НСУ-3 + БВС-4 (окно, контекст)"]])}.items()]
print("только тики с V < 1 км:"); display(pd.DataFrame(rows).set_index("метод").round(3))

In [ ]:
full_name = "БВС-5 + НСУ-3 + БВС-4 (окно, контекст)"; full_m = models[full_name]; fc = sets[full_name]
rs = np.random.default_rng(0); low = np.where(np.exp(yte) < 3000)[0]; idx = rs.choice(low, min(5000, len(low)), replace=False)
pi = permutation_importance(full_m, Xte[idx][:, fc], yte[idx], n_repeats=3, random_state=0, scoring="neg_median_absolute_error")
imp = pd.Series(pi.importances_mean, index=[VIS_FEATS[i] for i in fc]).sort_values().tail(14); imp.plot.barh(figsize=(5.6, 4.4)); plt.xlabel("рост медианной ошибки ln V (V < 3 км)"); plt.tight_layout(); plt.show()

## 3. События «видимость < 1 км» и «< 0,3 км»
Оценка сглаживается (EMA 10 с, лог-шкала); тревога - после 3 тиков ниже порога, отбой - выше 1,5 порога. Метрики: recall, медианная задержка от начала события, ложные тревоги в час.

In [ ]:
def pred(model, cols, X): return model.predict(X[:, cols])
methods = {"НСУ-3 (как есть)": lambda X: X[:, JN], "физическое слияние": lambda X: physics_fusion(X), "GBM: БВС-5": lambda X: pred(models["БВС-5"], sets["БВС-5"], X),
           "GBM: БВС-5 + НСУ-3": lambda X: pred(models["БВС-5 + НСУ-3"], sets["БВС-5 + НСУ-3"], X), "GBM: + БВС-4": lambda X: pred(full_m, fc, X)}
rows = []
for thr in (1000, 300):
    for n, f in methods.items(): rows.append(dict(порог_м=thr, метод=n, **events_vis(f(Xte), mte, test, thr)))
ev = pd.DataFrame(rows).set_index(["порог_м", "метод"]); display(ev.round(2))

## 4. Смена условий
Три нарушения предположений, на которых обучена модель (обучение - один раз, в исходных условиях): (а) связь влажности с видимостью ослаблена до 0,3 (события почти не меняют T и RH); (б) дальности lidar и камеры в 0,7 раза короче предполагаемых; (в) ошибка прогноза НСУ-3 больше (σ = 1,0).

In [ ]:
def make(label, params=None, modify=None):
    old = (st.LIDAR_K, st.CAM_K, st.NSU_SIGMA)
    try:
        if params: st.LIDAR_K, st.CAM_K, st.NSU_SIGMA = params
        ch = build_cache4(KINDS4, range(500, 505), modify=modify)
    finally:
        st.LIDAR_K, st.CAM_K, st.NSU_SIGMA = old
    X, y, m = build_rows_vis(ch); return label, ch, X, y, m
shifts = [make("связь влажности ×0,3", modify=lambda cfg: setattr(cfg, "hum_coupling", 0.3)), make("дальности сенсоров ×0,7", params=(0.105, 0.28, 0.6)), make("σ прогноза НСУ-3 = 1,0", params=(0.15, 0.4, 1.0))]
rows = []; evs = []
for label, ch, X, y, m in shifts:
    for n, f in methods.items():
        bb = by_bin(f(X), y); r = dict(условие=label, метод=n, **{k: bb.get(k, np.nan) for k in VLAB[:3]})
        e = events_vis(f(X), m, ch, 1000); r.update(recall_1км=e["recall"], ложных_в_час=e["ложных_в_час"]); rows.append(r)
sh = pd.DataFrame(rows).set_index(["условие", "метод"]); display(sh.round(2))

## Время обработки
Замер на тик БВС-5 (1 Гц): извлечение признаков (всё окно БВС-4) + предсказание GBM; физическое слияние - арифметика.

In [ ]:
def timing(cache, n_scen=3):
    ts_f, ts_p = [], []
    for c_ in cache[:n_scen]:
        fe = VisFeatures(c_["epoch"])
        for m in c_["msgs"]:
            t1 = time.perf_counter_ns(); x = fe.feed(m); t2 = time.perf_counter_ns()
            if x is not None:
                t3 = time.perf_counter_ns(); full_m.predict(x[fc].reshape(1, -1)); t4 = time.perf_counter_ns(); ts_p.append((t4 - t3) / 1e3)
            ts_f.append((t2 - t1) / 1e3)
    a, b = np.array(ts_f), np.array(ts_p)
    return pd.DataFrame({"извлечение признаков (все сообщения)": [np.percentile(a, 50), np.percentile(a, 99), a.max()], "предсказание GBM (тик БВС-5)": [np.percentile(b, 50), np.percentile(b, 99), b.max()]}, index=["p50_мкс", "p99_мкс", "max_мкс"]).T
tm = timing(test); tm["p99_мс при замедлении ×50"] = (tm.p99_мкс * 50 / 1000).round(2); display(tm.round(1))

## Выход модуля
Оценка видимости, интервал неопределённости, класс и **доверие сенсорам** (из предполагаемой дальности сенсора `min(R_max, k·V̂)`; k - калибровка конкретного аппарата).

In [ ]:
def visibility_class(V):
    return "clear" if V > 10000 else "haze" if V > 3000 else "reduced" if V > 1000 else "poor" if V > 300 else "very_poor"
def visibility_alert(t, V_hat, sigma_ln=0.5, k_l=0.15, k_c=0.4, uav_id="UAV-001"):
    lo, hi = V_hat * np.exp(-1.64 * sigma_ln), V_hat * np.exp(1.64 * sigma_ln); cls = visibility_class(V_hat)
    trust = {"lidar": round(float(np.clip(k_l * V_hat / 150.0, 0, 1)), 2), "camera": round(float(np.clip(k_c * V_hat / 120.0, 0, 1)), 2), "radar": 1.0}
    act = {"clear": "none", "haze": "monitor", "reduced": "lower_camera_weight; prefer_radar_for_avoidance", "poor": "rely_on_radar; slow_down; consider_altitude_change",
           "very_poor": "rely_on_radar_only; slow_down_or_land; request_route_replan(БВС-10)"}[cls]
    return {"alert_id": f"vis-{int(t * 1000)}", "uav_id": uav_id, "alert_type": None, "severity": {"clear": 0, "haze": 1, "reduced": 2, "poor": 3, "very_poor": 4}[cls], "description": "VISIBILITY_" + cls.upper(),
            "visibility_est_m": int(V_hat), "visibility_90ci_m": [int(lo), int(hi)], "sensor_trust": trust, "sensor_source": "NSU3+BVS4+BVS5", "timestamp": int(t * 1000), "recommended_action": act}
display(visibility_alert(210.0, 450.0))

## 9. Выводы и ограничения
**Источники по отдельности.**
- **НСУ-3 `visibility_m` как есть непригоден для локальных событий:** он завышает видимость в ≈ 18 раз при V < 0,3 км, в ≈ 6 раз при 0,3-1 км и в ≈ 4 раза при 1-3 км (прогноз сглаживает событие к фоновому значению). Событие «V < 1 км» он находит лишь в 21 % случаев, «V < 0,3 км» - ни разу. Простая линейная калибровка прогноза проблему не решает: ошибка зависит от события, а не от масштаба.
- **Физическая оценка по дальностям сенсоров** точна лишь в узком окне (при V ≈ 0,3-1 км отношение оценки к истине ≈ 1); при V > 3 км она занижает видимость, а при V < 0,3 км слепые сенсоры часто не дают первого обнаружения вовсе. Доступна она эпизодически: свежий трек препятствия есть в ≈ 76 % тиков при V ≈ 0,3-1 км, но лишь в 5-16 % тиков в других диапазонах. Её нельзя использовать без шлюзования и калибровки сенсора (`k` у lidar и камеры).

**Слияние (медиана |ln(V̂/V)| по диапазонам, среднее по 5 диапазонам).** НСУ-3 1,50; физическое слияние 1,22; бустинг только по БВС-5 0,57; БВС-5 + НСУ-3 **0,25**; БВС-5 + НСУ-3 + БВС-4 **0,22**; добавление дальностей первого обнаружения прироста не дало (0,24). На тиках с V < 1 км: 0,33 (×1,4) и 60 % оценок в пределах ×1,5 для полной модели против 0,3 % у НСУ-3.

**События.** «V < 1 км» (14 событий): бустинг по БВС-5 0,93; БВС-5 + НСУ-3 0,93 (задержка ≈ 17 с, 0,2 ложных/ч); с БВС-4 0,86 (0 ложных). «V < 0,3 км» (8 событий): БВС-5 0,62; **БВС-5 + НСУ-3 и с БВС-4: 1,0, задержка ≈ 24-26 с, 0 ложных**. Задержку определяют сглаживание (10 с + 3 тика) и период НСУ-3 (≈ 6 с).

**Смена условий (главный результат).**
- Связь влажности с видимостью ослаблена (×0,3): модель только по БВС-5 разваливается (ошибка в тумане ≈ 200× вместо 5×, события 0 %), БВС-5 + НСУ-3 держится на 0,67, **с признаками БВС-4 - 1,0 при 0 ложных**; физическое слияние на этом сдвиге тоже работает (0,89), но даёт ≈ 1 ложную тревогу в час.
- Сенсоры слепнут раньше (дальности ×0,7): с БВС-4 recall 1,0, 0 ложных; физическое слияние ломается на сильном тумане (ошибка 2,7 - неверная калибровка `k`).
- Прогноз НСУ-3 хуже (σ 1,0): с БВС-4 recall 1,0, 0 ложных; без БВС-4 - 0,89 при 0,4 ложных/ч.
Вывод: опираться на **несколько независимых источников** важнее, чем на точность каждого; БВС-4 даёт устойчивость к нарушению допущений, на которых обучены остальные.

**Время.** Извлечение признаков - p50 ≈ 5 мкс, p99 ≈ 143 мкс на сообщение (редкие выбросы ≈ 1 мс); бустинг построчно: p50 ≈ 1,1 мс, p99 ≈ 1,9 мс (при замедлении ×50 ≈ 93 мс) - укладывается в период БВС-5 (1 с), но не в быстрый контур ≲ 5 мс; на борту модель нужно сжать или вызывать асинхронно.

**Ограничения.**
- Связь дальностей сенсоров с видимостью (`0,15·V`, `0,4·V`), форма ошибки НСУ-3 и связь влажности с видимостью - допущения генератора; реальные k нужно калибровать по аппарату (на лётных данных с известной видимостью).
- Задержка по БВС-4 определяется частотой препятствий (ноутбук 02); в пустом воздушном пространстве оценки по дальностям недоступны.
- Модель обучена на сценариях, где ясная погода занимает 84 % времени; по диапазонам V < 1 км обучающих данных мало (≈ 5 % тиков), поэтому оценки неопределённости (90 %-й интервал в алерте) приближённые.
- Прямое измерение видимости (рассеяние вперёд), PM-датчики и видео в ТЗ отсутствуют; различить пыль, дым и туман по видимости нельзя, это делает классификатор условий (ноутбуки 01, 02).

**Дальше:** итоговый ноутбук интеграции (05): индекс пригодности условий и доверие сенсорам из всех детекторов (конденсат, метео, БВС-4, мощность/град, видимость) на смешанных сценариях со сквозным замером задержек (≤ 200 мс на сообщение, 1-5 с на причинные события) и ложных тревог в час.